# ICBHI AST-SAM — Colab Training Notebook

**Workflow:**
1. Mount Google Drive (persistent checkpoints survive disconnects)
2. Clone / pull your GitHub repo
3. (Optional) Open a VS Code tunnel to edit code from your local IDE
4. Install dependencies
5. Preprocess data (once) or skip if already done
6. Train with `--resume` so you pick up from the last saved epoch
7. Evaluate

> **Checkpoint strategy:** `latest_checkpoint.pt` is saved to Google Drive after
> every epoch. When Colab disconnects and you re-run, `python train.py --resume`
> automatically picks up from the last completed epoch.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────
import torch
print(f"GPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# All persistent files live here — survives runtime resets
DRIVE_PROJECT = '/content/drive/MyDrive/ICBHI-AST-SAM'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/results',     exist_ok=True)
print(f'Drive project folder: {DRIVE_PROJECT}')

In [ ]:
# ── Cell 3: Clone / pull repo ─────────────────────────────────────────
# Replace with your actual repo URL
REPO_URL = 'https://github.com/ajammoussi/SAM-Optimized-AST-Respiratory-Sound-Classification.git'
REPO_DIR = '/content/ICBHI-AST-SAM'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!ls -la

In [ ]:
# ── Cell 4 (Optional): VS Code Remote Tunnel ──────────────────────────
# This lets you edit files from your local VS Code while the GPU runs here.
# Requires: VS Code + "Remote - Tunnels" extension installed locally.
#
# After running this cell:
#   1. Copy the GitHub auth code that appears in the output.
#   2. Go to https://github.com/login/device and paste it.
#   3. In VS Code: Ctrl+Shift+P → 'Remote Tunnels: Connect to Tunnel'
#      → select the tunnel name you set below.

TUNNEL_NAME = 'colab-icbhi'   # change to anything memorable

!pip install vscode-colab -q
import vscode_colab
vscode_colab.connect(
    name=TUNNEL_NAME,
    git_user_name='Your Name',          # optional, for git commits
    git_user_email='you@example.com',   # optional
)

In [ ]:
# ── Cell 5: Install dependencies ──────────────────────────────────────
!pip install -r requirements.txt -q

In [ ]:
# ── Cell 6: Symlink checkpoints & results to Drive ────────────────────
# This means train.py writes directly to Drive without any manual copying.
import subprocess

LOCAL_CKPT    = f'{REPO_DIR}/checkpoints'
LOCAL_RESULTS = f'{REPO_DIR}/results'
DRIVE_CKPT    = f'{DRIVE_PROJECT}/checkpoints'
DRIVE_RESULTS = f'{DRIVE_PROJECT}/results'

for local, drive_path in [(LOCAL_CKPT, DRIVE_CKPT), (LOCAL_RESULTS, DRIVE_RESULTS)]:
    if os.path.islink(local):
        os.remove(local)
    elif os.path.isdir(local):
        import shutil; shutil.rmtree(local)
    os.symlink(drive_path, local)
    print(f'Linked {local} → {drive_path}')

In [ ]:
# ── Cell 7: Preprocess (run ONCE, skip if .npz already exists on Drive) ─
NPZ_PATH = f'{DRIVE_PROJECT}/icbhi_ast_16k_8s_spectrograms.npz'
DATA_DIR = f'{DRIVE_PROJECT}/data/ICBHI_final_database'
SPLIT    = f'{DRIVE_PROJECT}/data/ICBHI_challenge_train_test.txt'

if os.path.exists(NPZ_PATH):
    print(f'Pre-computed spectrograms found: {NPZ_PATH}  — skipping preprocessing.')
else:
    print('Running preprocessing (this takes ~5-10 min) …')
    !python preprocess.py \
        --data_dir  $DATA_DIR \
        --split_file $SPLIT \
        --output    $NPZ_PATH

# Symlink the npz into the repo dir so train.py finds it at its default path
LOCAL_NPZ = f'{REPO_DIR}/icbhi_ast_16k_8s_spectrograms.npz'
if not os.path.exists(LOCAL_NPZ):
    os.symlink(NPZ_PATH, LOCAL_NPZ)
    print(f'Linked {LOCAL_NPZ} → {NPZ_PATH}')

In [ ]:
# ── Cell 8: Train  (add --resume to continue after a disconnect) ───────
# Remove --resume on the very first run.
!python train.py \
    --data_path    ./icbhi_ast_16k_8s_spectrograms.npz \
    --checkpoint_dir ./checkpoints \
    --results_dir  ./results \
    --epochs       20 \
    --batch_size   16 \
    --lr           1e-5 \
    --looksam_k    5 \
    --num_workers  2 \
    --scheduler_t0 5 \
    --resume

In [ ]:
# ── Cell 9: Evaluate ──────────────────────────────────────────────────
!python evaluate.py \
    --data_path  ./icbhi_ast_16k_8s_spectrograms.npz \
    --model_path ./checkpoints/best_model.pth \
    --output_dir ./results

In [ ]:
# ── Cell 10: Inline preview of figures ────────────────────────────────
from IPython.display import display, Image
import glob

for fig_path in sorted(glob.glob('./results/*.png')):
    print(fig_path)
    display(Image(fig_path, width=800))